# spark.read — a JSON feed from an API (try it live)

Read a real API feed off disk with a schema you declare yourself.

Companion to the note: **spark.read — a JSON feed from an API**
on [ravi-writes.pages.dev](https://ravi-writes.pages.dev/playground/read-json-api-feed).

Run top to bottom: **Runtime → Run all**.

## The data & the requirement

[CoinGecko](https://www.coingecko.com/)'s public `coins/markets` endpoint returns one
record per cryptocurrency — id, name, price, market cap, rank, and 21 other columns.
We pull the **top 500 coins** and land them as a JSON file, then Spark reads that file.

Landing the file first with plain Python is the rule for any file example: the input has
to arrive from *outside* Spark, or reading it with Spark proves nothing. Here the
"outside" is a real HTTP API instead of Faker.

`per_page` maxes out at 250, so 500 coins is **two** requests. CoinGecko's free tier
rate-limits by IP and Colab hands you a *shared* one, so the live call sometimes returns
`HTTP 429` through no fault of yours — we try **three times** with backoff, then fall back
to a saved snapshot kept in the repo.

In [1]:
import json, os, time, urllib.parse, urllib.request

INPUT_PATH = "/content/ravi-writes/data/input/markets.json"
os.makedirs(os.path.dirname(INPUT_PATH), exist_ok=True)

SNAPSHOT_URL = ("https://raw.githubusercontent.com/ravi-asati/"
                "ravi-writes-abinitio-to-spark/main/datasets/coingecko-markets-500.json")


def cg(path, **params):
    """Call any CoinGecko v3 endpoint. Returns parsed JSON."""
    url = f"https://api.coingecko.com/api/v3/{path}"
    if params:
        url += "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "learning/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())


def fetch_live(pages=2, per_page=250):
    rows = []
    for page in range(1, pages + 1):
        rows += cg("coins/markets", vs_currency="usd", order="market_cap_desc",
                   per_page=per_page, page=page, sparkline="false")
        time.sleep(2)          # public tier is rate-limited — pause between pages
    return rows


rows = None
for attempt in range(1, 4):                       # 3 attempts
    try:
        rows = fetch_live()
        print(f"live API ok — {len(rows)} coins")
        break
    except Exception as e:
        print(f"  ! attempt {attempt}/3 failed: {e}")
        time.sleep(5 * attempt)                   # 5s, 10s, 15s

if rows is None:                                  # rate-limited or offline
    print("falling back to the saved snapshot")
    urllib.request.urlretrieve(SNAPSHOT_URL, INPUT_PATH)
else:
    with open(INPUT_PATH, "w") as f:
        json.dump(rows, f, indent=1)              # same shape as the snapshot

live API ok — 500 coins


In [41]:
!ls -la /content/ravi-writes/data/input
!head -50 /content/ravi-writes/data/input/markets.json

!echo '{"enteries":[{"id": 1, "name": "Ravi Asati"},{"id": 2, "name": "Aarti Asati"}]}' > /content/ravi-writes/data/input/markets_sample.json
!ls -la /content/ravi-writes/data/input/markets_sample.json
!head -5 /content/ravi-writes/data/input/markets_sample.json

!echo -n '{"enteries":[{"id": 1, "name": "Ravi Asati"},{"id": 2, "name": "Aarti Asati"}]}' > /content/ravi-writes/data/input/markets_sample_no_nl.json
!ls -la /content/ravi-writes/data/input/markets_sample_no_nl.json
!head -5 /content/ravi-writes/data/input/markets_sample_no_nl.json
!wc -l /content/ravi-writes/data/input/markets_sample_no_nl.json

!echo -n '[{"id": 1, "name": "Ravi Asati"},{"id": 2, "name": "Aarti Asati"}]' > /content/ravi-writes/data/input/markets_sample_v3.json
!ls -la /content/ravi-writes/data/input/markets_sample_v3.json
!head -5 /content/ravi-writes/data/input/markets_sample_v3.json
!wc -l /content/ravi-writes/data/input/markets_sample_v3.json

!echo -n '{"test":"{"id": 1, "name": "Ravi Asati"},{"id": 2, "name": "Aarti Asati"}"}' > /content/ravi-writes/data/input/markets_sample_v4.json
!ls -la /content/ravi-writes/data/input/markets_sample_v4.json
!head -5 /content/ravi-writes/data/input/markets_sample_v4.json
!wc -l /content/ravi-writes/data/input/markets_sample_v4.json

total 464
drwxr-xr-x 2 root root   4096 Aug 13 19:47 .
drwxr-xr-x 3 root root   4096 Aug 13 18:32 ..
-rw-r--r-- 1 root root 449828 Aug 13 18:32 markets.json
-rw-r--r-- 1 root root     80 Aug 13 19:47 markets_sample.json
-rw-r--r-- 1 root root     79 Aug 13 19:47 markets_sample_no_nl.json
-rw-r--r-- 1 root root     66 Aug 13 19:47 markets_sample_v3.json
-rw-r--r-- 1 root root     66 Aug 13 19:47 markets_sample_v4.json
[
 {
  "id": "bitcoin",
  "symbol": "btc",
  "name": "Bitcoin",
  "image": "https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400",
  "current_price": 63082,
  "market_cap": 1265971013392,
  "market_cap_rank": 1,
  "fully_diluted_valuation": 1265971013392,
  "total_volume": 19519134719,
  "high_24h": 63918,
  "low_24h": 62819,
  "price_change_24h": -223.78742134889762,
  "price_change_percentage_24h": -0.5,
  "market_cap_change_24h": -4530704211.30249,
  "market_cap_change_percentage_24h": -0.35661,
  "circulating_supply": 20069696.0,
  "total_suppl

Either way you end up with the same thing on disk: **one JSON file, 500 records,
pretty-printed as an array**.

### ⚠ Four things about this file that will bite you

They are all **real** — this is what the endpoint actually returns, not planted mess:

1. It is a **pretty-printed JSON array**, not one object per line. `spark.read.json`
   assumes one-object-per-line by default.
2. **`roi` is `null` for 463 of the 500 coins** and a nested object
   `{times, currency, percentage}` for the other 37.
3. **Prices arrive as both `int` and `float`** — Bitcoin's `current_price` is `63521`,
   most others look like `0.9998`. Same column, two JSON types.
4. **`max_supply` is null for 207 coins**, and `last_updated` / `ath_date` are ISO-8601
   **strings** (`"2026-08-13T03:33:20.000Z"`), not timestamps.

### The requirement

Read that file into a DataFrame with a schema **you declare** — not one Spark guesses.
Keep these eight columns:

`id` · `symbol` · `name` · `current_price` · `market_cap` · `market_cap_rank` ·
`total_volume` · `last_updated`

with `current_price` and `total_volume` as `DoubleType`, `market_cap` wide enough not to
overflow, `market_cap_rank` as an integer, and `last_updated` as a **real timestamp** —
not text. Then show the **top 10 coins by market cap**.

## The Ab Initio solution

_(to be written)_

## The Spark solution — step by step

In [5]:
# 1. Install PySpark
!pip install -q pyspark

In [6]:
# 2. Imports & SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType,
    LongType, TimestampType,
)

spark = SparkSession.builder.appName("read-json-api-feed").getOrCreate()

In [18]:
# 3. Your solution — declare the schema and read the file
#    (see the requirement above)
load_options = {
    "multiLine": True
                }

json_DF = spark.read.format("json").options(**load_options).load("/content/ravi-writes/data/input/markets.json")

# Display the DataFrame
json_DF.show(10)

json_DF.printSchema(1)

+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+-------------+-----------------------+--------+------------+--------------------+--------------------+--------+-------------+---------------------+--------------------------------+---------------+----------+------------+--------------------+---------------------------+--------------------+----------+--------------------+---------------+
|     ath|ath_change_percentage|            ath_date|       atl|atl_change_percentage|            atl_date|  circulating_supply|current_price|fully_diluted_valuation|high_24h|          id|               image|        last_updated| low_24h|   market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|        name|    price_change_24h|price_change_percentage_24h|                 roi|    symbol|        total_supply|   total_volume|
+--------+---------------------+--------------------+----------+--

In [32]:
# 3. Your solution — declare the schema and read the file
#    (see the requirement above)
load_options = {
    "multiLine": False
                }

sample_json_DF = spark.read.format("json").options(**load_options).load("/content/ravi-writes/data/input/markets_sample.json")

# Display the DataFrame
#sample_json_DF.show(10)

sample_json_DF.show(truncate = False)

sample_json_DF.printSchema(3)

+-----------------------------------+
|enteries                           |
+-----------------------------------+
|[{1, Ravi Asati}, {2, Aarti Asati}]|
+-----------------------------------+

root
 |-- enteries: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)



In [35]:
# 3. Your solution — declare the schema and read the file
#    (see the requirement above)
load_options = {
    "multiLine": True
                }

sample_no_nl_json_DF = spark.read.format("json").options(**load_options).load("/content/ravi-writes/data/input/markets_sample_no_nl.json")

# Display the DataFrame
#sample_json_DF.show(10)

sample_no_nl_json_DF.show(truncate = False)

sample_no_nl_json_DF.printSchema(3)

+-----------------------------------+
|enteries                           |
+-----------------------------------+
|[{1, Ravi Asati}, {2, Aarti Asati}]|
+-----------------------------------+

root
 |-- enteries: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)



In [37]:
load_options = {
    "multiLine": True
                }

sample_v3_json_DF = spark.read.format("json").options(**load_options).load("/content/ravi-writes/data/input/markets_sample_v3.json")

# Display the DataFrame
#sample_json_DF.show(10)

sample_v3_json_DF.show(truncate = False)

sample_v3_json_DF.printSchema(3)

+---+-----------+
|id |name       |
+---+-----------+
|1  |Ravi Asati |
|2  |Aarti Asati|
+---+-----------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [42]:
load_options = {
    "multiLine": False
                }

sample_v4_json_DF = spark.read.format("json").options(**load_options).load("/content/ravi-writes/data/input/markets_sample_v4.json")

# Display the DataFrame
#sample_json_DF.show(10)

sample_v4_json_DF.show(truncate = False)

sample_v4_json_DF.printSchema(3)

AnalysisException: [UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN] The feature is not supported: Queries from raw JSON/CSV/XML files are disallowed when the
referenced columns only include the internal corrupt record column
(named `_corrupt_record` by default). For example:
`spark.read.schema(schema).json(file).filter($"_corrupt_record".isNotNull).count()`
and `spark.read.schema(schema).json(file).select("_corrupt_record").show()`.
Instead, you can cache or save the parsed results and then send the same query.
For example, `val df = spark.read.schema(schema).json(file).cache()` and then
`df.filter($"_corrupt_record".isNotNull).count()`. SQLSTATE: 0A000

## Your turn

Variations to try yourself (no solutions provided):

1. **Let Spark guess instead.** Read the same file with no `.schema(...)` and compare
   `printSchema()` against your explicit one. What type did it pick for `current_price`,
   and what did it do with `roi`?
2. **Break it on purpose.** Read the file *without* the option that handles a multi-line
   array. How many rows come back, and what shows up in `_corrupt_record`?
3. **Keep the nested column.** Add `roi` to your schema as a struct and pull `roi.times`
   out as a top-level column. What happens on the 463 coins where it is null?
4. **Land it as Parquet.** Write the result out with `df.write`, read it back, and check
   `printSchema()` — did your timestamp survive the round trip? Why does Parquet not need
   a schema on the way back in?